# 02 — Silver Cleaning and Data Quality
Standardizes types, removes exact duplicate rows, detects conflicting timestamps, applies reusable plausibility rules, and separates valid records from quarantined records.

**Input:** `pgcb.bronze.grid_raw`  
**Outputs:**
- `pgcb.silver.grid_clean`
- `pgcb.silver.grid_quarantine`

The thresholds below are **project-defined data-quality rules**, not official PGCB operating limits.

In [ ]:
from functools import reduce
from operator import or_
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "pgcb.bronze.grid_raw"
SILVER_TABLE = "pgcb.silver.grid_clean"
QUARANTINE_TABLE = "pgcb.silver.grid_quarantine"

bronze_df = spark.table(BRONZE_TABLE)
print(f"Bronze rows: {bronze_df.count():,}")

## 1. Remove exact duplicate rows and standardize timestamp

In [ ]:
df = bronze_df.dropDuplicates()

print(f"Rows after removing exact duplicates: {df.count():,}")

df = df.withColumn("datetime_clean", F.to_timestamp("datetime"))

## 2. Convert numeric fields safely

In [ ]:
numeric_cols = [
    "generation_mw", "demand_mw", "load_shedding",
    "gas", "liquid_fuel", "coal", "hydro", "solar", "wind",
    "india_bheramara_hvdc", "india_tripura", "india_adani", "nepal"
]

NULL_TOKENS = ["nan", "na", "null", ""]

for c in numeric_cols:
    raw_string = F.trim(F.col(c).cast("string"))
    df = df.withColumn(
        f"{c}_clean",
        F.when(F.lower(raw_string).isin(NULL_TOKENS), F.lit(None).cast("double"))
         .otherwise(F.col(c).cast("double"))
    )

# Flag malformed non-null numeric values without treating source NULLs as errors.
conversion_error_exprs = []
for c in numeric_cols:
    raw_string = F.trim(F.col(c).cast("string"))
    conversion_error_exprs.append(
        F.col(c).isNotNull()
        & (~F.lower(raw_string).isin(NULL_TOKENS))
        & F.col(f"{c}_clean").isNull()
    )

df = df.withColumn(
    "numeric_conversion_error",
    reduce(or_, conversion_error_exprs)
)

## 3. Detect conflicting timestamps

In [ ]:
# Exact duplicate rows were already removed. If a timestamp still appears more than once,
# the source contains conflicting observations. These rows are quarantined rather than
# choosing one arbitrarily.
window_by_timestamp = Window.partitionBy("datetime_clean")

df = df.withColumn(
    "timestamp_record_count",
    F.count(F.lit(1)).over(window_by_timestamp)
).withColumn(
    "duplicate_timestamp",
    F.col("datetime_clean").isNotNull() & (F.col("timestamp_record_count") > 1)
)

## 4. Apply project-defined plausibility rules

In [ ]:
# Bounds are derived from the original academic cleaning logic and are intentionally
# centralized here so they can be reviewed or replaced later.
BOUNDS = {
    "generation_mw": (3000, 17000),
    "demand_mw": (1000, 20000),
    "load_shedding": (0, 10000),
    "gas": (0, 16000),
    "liquid_fuel": (0, 10000),
    "coal": (0, 5000),
    "hydro": (0, 300),
    "wind": (0, 100),
    "india_bheramara_hvdc": (0, 2000),
    "india_tripura": (0, 200),
    "india_adani": (0, 2000),
}

quality_flag_cols = []
for c, (lower, upper) in BOUNDS.items():
    flag_col = f"{c}_invalid"
    quality_flag_cols.append(flag_col)
    df = df.withColumn(
        flag_col,
        F.when(
            F.col(f"{c}_clean").isNotNull()
            & ((F.col(f"{c}_clean") < lower) | (F.col(f"{c}_clean") > upper)),
            F.lit(True)
        ).otherwise(F.lit(False))
    )

# Core fields must exist for the forecasting/operations use case.
df = df.withColumn(
    "core_field_missing",
    F.col("datetime_clean").isNull()
    | F.col("generation_mw_clean").isNull()
    | F.col("demand_mw_clean").isNull()
)

quality_issue_expr = (
    F.col("core_field_missing")
    | F.col("numeric_conversion_error")
    | F.col("duplicate_timestamp")
    | reduce(or_, [F.col(c) for c in quality_flag_cols])
)

df = df.withColumn("has_quality_issue", quality_issue_expr)

df.groupBy("has_quality_issue").count().show()

## 5. Split valid and quarantine records

In [ ]:
valid_df = df.filter(~F.col("has_quality_issue"))
quarantine_df = df.filter(F.col("has_quality_issue"))

spark.sql("CREATE SCHEMA IF NOT EXISTS pgcb.silver")

(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(QUARANTINE_TABLE)
)

## 6. Create clean analytical fields

In [ ]:
valid_df = (
    valid_df
    .withColumn("date", F.to_date("datetime_clean"))
    .withColumn("year", F.year("datetime_clean"))
    .withColumn("month", F.month("datetime_clean"))
    .withColumn("day", F.dayofmonth("datetime_clean"))
    .withColumn("hour", F.hour("datetime_clean"))
    .withColumn("day_of_week", F.dayofweek("datetime_clean"))
)

# Component sums use zero only for arithmetic. The underlying source columns retain NULL,
# so missing/not-reported values remain visible to downstream users.
valid_df = valid_df.withColumn(
    "domestic_generation_mw",
    sum(F.coalesce(F.col(f"{c}_clean"), F.lit(0.0)) for c in [
        "gas", "liquid_fuel", "coal", "hydro", "solar", "wind"
    ])
).withColumn(
    "total_imports_mw",
    sum(F.coalesce(F.col(f"{c}_clean"), F.lit(0.0)) for c in [
        "india_bheramara_hvdc", "india_tripura", "india_adani", "nepal"
    ])
)

valid_df = valid_df.withColumn(
    "calculated_supply_mw",
    F.col("domestic_generation_mw") + F.col("total_imports_mw")
).withColumn(
    "supply_demand_gap_mw",
    F.col("generation_mw_clean") - F.col("demand_mw_clean")
)

## 7. Select and write the Silver table

In [ ]:
silver_df = valid_df.select(
    F.col("datetime_clean").alias("datetime"),
    F.col("generation_mw_clean").alias("generation_mw"),
    F.col("demand_mw_clean").alias("demand_mw"),
    F.col("load_shedding_clean").alias("load_shedding"),
    F.col("gas_clean").alias("gas"),
    F.col("liquid_fuel_clean").alias("liquid_fuel"),
    F.col("coal_clean").alias("coal"),
    F.col("hydro_clean").alias("hydro"),
    F.col("solar_clean").alias("solar"),
    F.col("wind_clean").alias("wind"),
    F.col("india_bheramara_hvdc_clean").alias("india_bheramara_hvdc"),
    F.col("india_tripura_clean").alias("india_tripura"),
    F.col("india_adani_clean").alias("india_adani"),
    F.col("nepal_clean").alias("nepal"),
    "remarks",
    "date", "year", "month", "day", "hour", "day_of_week",
    "domestic_generation_mw", "total_imports_mw", "calculated_supply_mw",
    "supply_demand_gap_mw", "_source_file", "_ingested_at"
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

## 8. Validate row counts

In [ ]:
bronze_count = spark.table(BRONZE_TABLE).count()
silver_count = spark.table(SILVER_TABLE).count()
quarantine_count = spark.table(QUARANTINE_TABLE).count()

print(f"Bronze rows:     {bronze_count:,}")
print(f"Silver rows:     {silver_count:,}")
print(f"Quarantine rows: {quarantine_count:,}")

display(spark.table(SILVER_TABLE).limit(20))